In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output, State
import plotly.express as px
import base64
import pandas as pd
import re

JupyterDash.infer_jupyter_proxy_config()

# Import the enhanced CRUD module
from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "cs-340"

# Connect to MongoDB through the CRUD module
try:
    db = AnimalShelter(username, password)
except Exception as error:
    print(f"FATAL ERROR: Could not connect to MongoDB. Details: {error}")
    db = None

# Return only the fields that the dashboard uses.
# This avoids sending MongoDB's ObjectId to Dash and reduces unnecessary data.
projection = {
    "_id": 0,
    "animal_id": 1,
    "name": 1,
    "animal_type": 1,
    "breed": 1,
    "sex_upon_outcome": 1,
    "age_upon_outcome_in_weeks": 1,
    "outcome_type": 1,
    "outcome_subtype": 1,
    "location_lat": 1,
    "location_long": 1
}

# Define the columns even when a search or query returns no records.
dashboard_columns = list(projection.keys())

# Load the complete projected dataset once.
# MongoDB performs the initial breed sort.
if db:
    try:
        all_animals = db.read(
            {},
            projection=projection,
            sort=[("breed", 1)]
        )
    except Exception as error:
        print(f"ERROR: Could not load the initial dataset. Details: {error}")
        all_animals = []
else:
    all_animals = []

# Build a HashMap for average O(1) animal ID lookup.
if db and hasattr(db, "build_animal_hashmap"):
    animal_hashmap = db.build_animal_hashmap(all_animals)
else:
    animal_hashmap = {}
    for animal in all_animals:
        animal_id = animal.get("animal_id")
        if animal_id:
            animal_hashmap.setdefault(animal_id, []).append(animal)

df = pd.DataFrame.from_records(all_animals, columns=dashboard_columns)

# Clean geolocation fields.
if not df.empty:
    df["location_long"] = pd.to_numeric(df["location_long"], errors="coerce")
    df["location_lat"] = pd.to_numeric(df["location_lat"], errors="coerce")


#################################
# Logo Encoding / Branding Setup
#################################

image_filename = "GraziosoSalvareLogo.png"

try:
    with open(image_filename, "rb") as image_file:
        encoded_image = base64.b64encode(image_file.read()).decode("ascii")
    logo_src = f"data:image/png;base64,{encoded_image}"
except FileNotFoundError:
    print(
        f"WARNING: Logo file '{image_filename}' was not found. "
        "A placeholder will be used."
    )
    logo_src = (
        "data:image/png;base64,"
        "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJ"
        "AAAADUlEQVR42mNk+M9QDwADhgGAWjRDAAAAAElFTkSuQmCC"
    )


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

filter_options = [
    {"label": "Water Rescue", "value": "water_rescue"},
    {"label": "Mountain or Wilderness Rescue", "value": "mountain_rescue"},
    {
        "label": "Disaster Rescue or Individual Tracking",
        "value": "disaster_rescue"
    },
    {"label": "All / No Filter", "value": "all"}
]

app.layout = html.Div([
    # Header
    html.Div([
        html.Img(
            src=logo_src,
            style={
                "height": "100px",
                "marginRight": "20px"
            }
        ),
        html.H1(
            "Grazioso Salvare Rescue Dashboard - Bruno DeSousa",
            style={"color": "#1E90FF", "margin": "0"}
        )
    ], style={
        "padding": "10px",
        "borderBottom": "2px solid #1E90FF",
        "display": "flex",
        "alignItems": "center",
        "justifyContent": "center"
    }),

    html.Hr(),

    # Rescue filter controls
    html.Div([
        html.H3("Select Rescue Type Filter:"),
        dcc.RadioItems(
            id="filter-type",
            options=filter_options,
            value="all",
            labelStyle={
                "display": "inline-block",
                "marginRight": "20px"
            }
        )
    ], style={"padding": "20px", "backgroundColor": "#f8f8f8"}),

    # HashMap animal ID search
    html.Div([
        html.H3("Search by Animal ID:"),
        dcc.Input(
            id="animal-id-search",
            type="text",
            placeholder="Example: A555413",
            debounce=True,
            style={"marginRight": "10px", "width": "220px"}
        ),
        html.Button(
            "Search",
            id="search-button",
            n_clicks=0,
            style={"marginRight": "10px"}
        ),
        html.Button(
            "Clear Search",
            id="clear-search-button",
            n_clicks=0
        ),
        html.Div(
            id="search-message",
            style={"marginTop": "10px", "fontWeight": "bold"}
        )
    ], style={"padding": "20px"}),

    html.Hr(),

    # Data table
    dash_table.DataTable(
        id="datatable-id",
        columns=[
            {
                "name": column,
                "id": column,
                "deletable": False,
                "selectable": True
            }
            for column in dashboard_columns
        ],
        data=df.to_dict("records"),
        page_current=0,
        page_size=10,
        page_action="native",
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[],
        style_header={
            "backgroundColor": "lightgrey",
            "fontWeight": "bold"
        },
        style_data_conditional=[
            {
                "if": {"row_index": "odd"},
                "backgroundColor": "rgb(248, 248, 248)"
            }
        ],
        style_table={"overflowX": "auto"},
        virtualization=True
    ),

    html.Br(),
    html.Hr(),

    # Chart and map
    html.Div(
        className="row",
        style={"display": "flex", "flexWrap": "wrap"},
        children=[
            html.Div(
                id="graph-id",
                className="col s12 m6",
                style={"width": "49%", "padding": "10px"}
            ),
            html.Div(
                id="map-id",
                className="col s12 m6",
                style={"width": "49%", "padding": "10px"}
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

def get_rescue_query(filter_type):
    """Return the MongoDB query for the selected rescue category."""

    if filter_type == "water_rescue":
        lab_regex = re.compile(".*lab.*", re.IGNORECASE)
        chesapeake_regex = re.compile(".*chesa.*", re.IGNORECASE)
        newfoundland_regex = re.compile(".*newf.*", re.IGNORECASE)

        return {
            "$or": [
                {"breed": {"$regex": newfoundland_regex}},
                {"breed": {"$regex": chesapeake_regex}},
                {"breed": {"$regex": lab_regex}}
            ],
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {
                "$gte": 26.0,
                "$lte": 156.0
            }
        }

    if filter_type == "mountain_rescue":
        german_regex = re.compile(".*german.*", re.IGNORECASE)
        malamute_regex = re.compile(".*mala.*", re.IGNORECASE)
        old_english_regex = re.compile(".*old english.*", re.IGNORECASE)
        husky_regex = re.compile(".*husk.*", re.IGNORECASE)
        rottweiler_regex = re.compile(".*rott.*", re.IGNORECASE)

        return {
            "$or": [
                {"breed": {"$regex": german_regex}},
                {"breed": {"$regex": malamute_regex}},
                {"breed": {"$regex": old_english_regex}},
                {"breed": {"$regex": husky_regex}},
                {"breed": {"$regex": rottweiler_regex}}
            ],
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26.0,
                "$lte": 156.0
            }
        }

    if filter_type == "disaster_rescue":
        german_regex = re.compile(".*german.*", re.IGNORECASE)
        golden_regex = re.compile(".*golden.*", re.IGNORECASE)
        bloodhound_regex = re.compile(".*blood.*", re.IGNORECASE)
        doberman_regex = re.compile(".*dober.*", re.IGNORECASE)
        rottweiler_regex = re.compile(".*rott.*", re.IGNORECASE)

        return {
            "$or": [
                {"breed": {"$regex": german_regex}},
                {"breed": {"$regex": golden_regex}},
                {"breed": {"$regex": bloodhound_regex}},
                {"breed": {"$regex": doberman_regex}},
                {"breed": {"$regex": rottweiler_regex}}
            ],
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20.0,
                "$lte": 300.0
            }
        }

    return {}


@app.callback(
    [
        Output("datatable-id", "data"),
        Output("datatable-id", "columns"),
        Output("datatable-id", "selected_rows"),
        Output("search-message", "children"),
        Output("animal-id-search", "value")
    ],
    [
        Input("filter-type", "value"),
        Input("search-button", "n_clicks"),
        Input("clear-search-button", "n_clicks")
    ],
    [State("animal-id-search", "value")]
)
def update_dashboard(
    filter_type,
    search_clicks,
    clear_clicks,
    animal_id
):
    """Update the table from either a rescue filter or animal ID search."""

    callback_context = dash.callback_context
    triggered_id = (
        callback_context.triggered[0]["prop_id"].split(".")[0]
        if callback_context.triggered
        else "filter-type"
    )

    # HashMap search
    if triggered_id == "search-button":
        if not animal_id or not animal_id.strip():
            message = "Enter an animal ID before searching."
            return [], _build_columns(), [], message, animal_id

        search_id = animal_id.strip().upper()

        if db and hasattr(db, "find_by_animal_id"):
            matching_records = db.find_by_animal_id(
                animal_hashmap,
                search_id
            )
        else:
            matching_records = animal_hashmap.get(search_id, [])

        # Python's sorted() uses Timsort.
        if db and hasattr(db, "sort_animals"):
            matching_records = db.sort_animals(
                matching_records,
                "breed"
            )
        else:
            matching_records = sorted(
                matching_records,
                key=lambda animal: str(
                    animal.get("breed", "")
                ).lower()
            )

        if matching_records:
            message = (
                f"Found {len(matching_records)} record(s) for "
                f"{search_id} using the animal ID HashMap."
            )
        else:
            message = f"No records were found for {search_id}."

        return (
            matching_records,
            _build_columns(),
            [],
            message,
            search_id
        )

    # Clear search and return to the currently selected rescue filter.
    if triggered_id == "clear-search-button":
        animal_id = ""

    query = get_rescue_query(filter_type)

    if not db:
        return (
            [],
            _build_columns(),
            [],
            "The dashboard is not connected to MongoDB.",
            animal_id
        )

    try:
        # MongoDB handles filtering, projection, and initial sorting.
        filtered_records = db.read(
            query,
            projection=projection,
            sort=[("breed", 1)]
        )

        message = (
            f"Displaying {len(filtered_records)} record(s)."
            if filtered_records
            else "No records match the selected rescue filter."
        )

        return (
            filtered_records,
            _build_columns(),
            [],
            message,
            animal_id
        )

    except Exception as error:
        print(
            f"ERROR: Failed to read data from MongoDB with "
            f"query {query}. Details: {error}"
        )
        return (
            [],
            _build_columns(),
            [],
            "An error occurred while loading the selected records.",
            animal_id
        )


def _build_columns():
    """Return a consistent Dash table column definition."""
    return [
        {
            "name": column,
            "id": column,
            "deletable": False,
            "selectable": True
        }
        for column in dashboard_columns
    ]


# Highlight the selected row.
@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_rows")]
)
def update_styles(selected_rows):
    styles = [
        {
            "if": {"row_index": "odd"},
            "backgroundColor": "rgb(248, 248, 248)"
        }
    ]

    if selected_rows:
        styles.extend([
            {
                "if": {"row_index": row_index},
                "backgroundColor": "#D2F3FF",
                "fontWeight": "bold"
            }
            for row_index in selected_rows
        ])

    return styles


# Update the chart using the currently displayed table records.
@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(view_data):
    if not view_data:
        return dcc.Graph(
            figure=px.bar(title="No data available for charting.")
        )

    chart_df = pd.DataFrame.from_dict(view_data)

    if (
        "outcome_subtype" not in chart_df.columns
        or chart_df["outcome_subtype"].dropna().empty
    ):
        return dcc.Graph(
            figure=px.bar(title="No outcome subtype data available.")
        )

    counts = (
        chart_df["outcome_subtype"]
        .fillna("Unknown")
        .value_counts()
        .nlargest(10)
        .reset_index()
    )
    counts.columns = ["Outcome Subtype", "Count"]

    figure = px.bar(
        counts,
        x="Outcome Subtype",
        y="Count",
        title="Top 10 Outcome Subtypes (Filtered)",
        color="Outcome Subtype",
        template="plotly_white"
    )

    return dcc.Graph(figure=figure)


# Update the map using the currently displayed and selected table row.
@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows")
    ]
)
def update_map(view_data, selected_rows):
    default_location = [30.75, -97.48]

    if not view_data:
        return _create_map(
            default_location,
            "Austin Animal Center",
            "Austin Animal Center",
            "Shelter Home Location"
        )

    selected_index = (
        selected_rows[0]
        if selected_rows and selected_rows[0] < len(view_data)
        else 0
    )

    selected_animal = view_data[selected_index]

    latitude = pd.to_numeric(
        selected_animal.get("location_lat"),
        errors="coerce"
    )
    longitude = pd.to_numeric(
        selected_animal.get("location_long"),
        errors="coerce"
    )

    if pd.isna(latitude) or pd.isna(longitude):
        return _create_map(
            default_location,
            "Austin Animal Center",
            "Location unavailable",
            "This animal does not have valid coordinates."
        )

    animal_location = [float(latitude), float(longitude)]
    breed = selected_animal.get("breed") or "Unknown breed"
    name = selected_animal.get("name") or "No name"

    return _create_map(
        animal_location,
        breed,
        "Animal Name",
        name
    )


def _create_map(center, tooltip, popup_heading, popup_text):
    """Create a Leaflet map centered on the supplied coordinates."""
    return [
        dl.Map(
            style={"width": "100%", "height": "500px"},
            center=center,
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=center,
                    children=[
                        dl.Tooltip(str(tooltip)),
                        dl.Popup([
                            html.H1(str(popup_heading)),
                            html.P(str(popup_text))
                        ])
                    ]
                )
            ]
        )
    ]


# Run the dashboard in JupyterLab.
app.run_server(mode="inline")